# 22b — Caveat Resolution and Quality Recode v2

This notebook fixes the v1 caveat-recode problem where all North West wards were incorrectly pushed into **Serious caveat / manual review**.

The v2 rule is simple: only mark a ward as serious where the evidence needed for interpretation is genuinely missing or invalid. Technical or medium-confidence issues, especially county-derived election evidence, are recoded as **medium confidence**, not hidden from the main report.

Outputs are written to:

```text
data/processed/caveat_resolution_v2/
```

In [1]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

INPUT_DIRS = [
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR / "target_model_v2",
    PROCESSED_DIR / "target_model_v1" / "inputs",
    PROCESSED_DIR / "target_model_v1" / "outputs",
    PROCESSED_DIR / "aggregations_v1",
    PROCESSED_DIR,
]

OUTPUT_DIR = PROCESSED_DIR / "caveat_resolution_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Override these manually if needed.
REVIEW_FILENAME = "north_west_consolidated_target_review_v1.csv"
MODEL_INPUT_FILENAME = "target_model_input_north_west_ward25_v1.csv"

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2


In [2]:
def find_file(filename, required=True):
    for folder in INPUT_DIRS:
        p = folder / filename
        if p.exists():
            return p
    # permissive recursive fallback
    for root in [PROCESSED_DIR, PROJECT_DIR]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(f"Could not find {filename}. Check INPUT_DIRS or set a manual path.")
    return None


def read_csv_file(filename, required=True):
    p = find_file(filename, required=required)
    if p is None:
        print("Optional file not found:", filename)
        return None, None
    df = pd.read_csv(p, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {p}")
    return df, p


def norm_bool(series):
    return series.fillna(False).astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


def is_nonempty_text(series):
    return series.notna() & series.astype(str).str.strip().ne("") & ~series.astype(str).str.lower().isin(["nan", "none", "null"])


def to_num(series):
    return pd.to_numeric(series, errors="coerce")

## 22b.1 Load the review table and recover missing quality fields

The consolidated review file used in Notebook 20 does not always contain all election-quality fields, especially valid-vote columns. This notebook therefore optionally merges fields from `target_model_input_north_west_ward25_v1.csv` where available.

In [3]:
review, review_path = read_csv_file(REVIEW_FILENAME)
model_input, model_input_path = read_csv_file(MODEL_INPUT_FILENAME, required=False)

review["WD25CD"] = review["WD25CD"].astype(str).str.strip()

# Merge useful quality columns from the model input if they are missing from the review table.
if model_input is not None and "WD25CD" in model_input.columns:
    model_input["WD25CD"] = model_input["WD25CD"].astype(str).str.strip()
    useful_cols = [
        "WD25CD",
        "latest_election_allocated_valid_votes",
        "latest_election_allocated_electorate",
        "latest_election_allocated_ballots",
        "latest_election_aggregation_label",
        "latest_election_latest_layer_note",
        "has_latest_election_layer",
        "has_valid_vote_data",
        "has_margin_data",
        "target_model_ready",
    ]
    useful_cols = [c for c in useful_cols if c in model_input.columns]
    enrich = model_input[useful_cols].drop_duplicates("WD25CD")
    before_cols = set(review.columns)
    review = review.merge(enrich, on="WD25CD", how="left", suffixes=("", "_model_input"), validate="many_to_one")
    print("Merged model-input quality fields:", sorted(set(review.columns) - before_cols))
else:
    print("No model-input enrichment applied.")

print("Review rows:", len(review))
display(review.head())

Loaded north_west_consolidated_target_review_v1.csv: (825, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_consolidated_target_review_v1.csv
Loaded target_model_input_north_west_ward25_v1.csv: (825, 119) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\inputs\target_model_input_north_west_ward25_v1.csv
Merged model-input quality fields: ['has_latest_election_layer', 'has_margin_data', 'has_valid_vote_data', 'latest_election_aggregation_label', 'latest_election_allocated_ballots', 'latest_election_allocated_electorate', 'latest_election_allocated_valid_votes', 'latest_election_latest_layer_note', 'target_model_ready']
Review rows: 825


,LAD25CD,LAD25NM,WD25CD,WD25NM,analysis_region,strategic_lane,strategic_lane_priority,strategic_lane_flags,is_clean_watchlist,is_caveated_watchlist,is_breakthrough_complacency,is_demographic_build,is_top100_watchlist,initial_watchlist_score,initial_watchlist_percentile,review_band,review_band_clean,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,data_confidence_score,dominant_cluster_name,second_cluster_name,latest_election_source_year,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_margin_pct_allocated,has_major_caveat,boundary_caveat,county_election_caveat,data_confidence_note,model_review_summary,candidate_known,candidate_name,candidate_strength_rating,local_contact_known,member_presence,recent_sdp_activity,local_issue_hook,activist_accessibility,delivery_practicality,campaign_cost_estimate,manual_priority,manual_notes,reviewed_by,reviewed_date,latest_election_allocated_valid_votes,latest_election_allocated_electorate,latest_election_allocated_ballots,latest_election_aggregation_label,latest_election_latest_layer_note,has_latest_election_layer,has_valid_vote_data,has_margin_data,target_model_ready
0,E07000121,Lancaster,E05014894,Heysham North,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,78.611324,99.617010,Review A,Review A Clean,81.165382,83.218786,62.951216,54.253246,95,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,2023.0,lab,independent,0.056387,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Sett...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,869.0,3427.0,0.0,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk,True,True,True,True
1,E06000009,Blackpool,E05015206,Waterloo,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,75.953840,98.930269,Review A,Review A Clean,82.480342,81.775542,52.212231,50.343514,95,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,2023.0,lab,con,0.031357,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Post...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1754.0,5155.0,0.0,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk,True,True,True,True
2,E08000004,Oldham,E05014652,Failsworth East,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,75.034628,98.335975,Review A,Review A Clean,70.661396,81.132744,63.853265,49.322090,100,Settled Working Families / Skilled Trades Suburbs,Rooted Older Homeowners,2024.0,lab,other,0.008601,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Sett...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2209.0,7905.0,0.0,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk,True,True,True,True
3,E08000001,Bolton,E05014823,Farnworth South,North West,Clean Opportunity,1,Clean Opportunity,True,False,False,False,True,74.769921,98.217116,Review A,Review A Clean,87.221810,66.303463,57.404996,0.000000,100,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,2024.0,other,lab,0.098498,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Post...,False,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2264.0,9601.0,0.0,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk,True,True,True,True
4,E06000009,Blackpool,E05015188,Bloomfield,North West,Clean Opportunity,1,Clean Opportunity; Breakthrough Build,True,False,True,False,True,73.799538,97.583201,Review A,Review A Clean,95.363090,69.017176,40.869214,68.561254,95,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,2023.0,lab,independent,0.136145,False,NaN,NaN,No major caveat.,Lane: Clean Opportunity | Dominant tribe: Post...,False,NaN,NaN,False,

## 22b.2 Recode caveats using actual available columns

Core logic:

- **High confidence** = election year, top party and margin exist, and no major caveat.
- **Medium confidence** = usable evidence but boundary/county-derived evidence requires a note.
- **Serious/manual review** = missing election layer, missing top party, missing margin, or explicit invalid vote data.

If `latest_election_allocated_valid_votes` is not present, the notebook will not treat that as invalid by itself.

In [4]:
df = review.copy()

# Field availability flags.
df["has_election_year_v2"] = "latest_election_source_year" in df.columns and to_num(df["latest_election_source_year"]).notna()
df["has_top_party_v2"] = "latest_election_top_party_bucket" in df.columns and is_nonempty_text(df["latest_election_top_party_bucket"])
df["has_margin_v2"] = "latest_election_margin_pct_allocated" in df.columns and to_num(df["latest_election_margin_pct_allocated"]).notna()

# Valid-vote logic: only mark invalid if the column exists and is <= 0, or if an existing data note explicitly says no usable valid vote data.
if "latest_election_allocated_valid_votes" in df.columns:
    valid_votes = to_num(df["latest_election_allocated_valid_votes"])
    df["has_valid_vote_data_v2"] = valid_votes.gt(0)
    df["valid_vote_data_unknown_v2"] = False
else:
    df["has_valid_vote_data_v2"] = True
    df["valid_vote_data_unknown_v2"] = True

if "has_valid_vote_data" in df.columns:
    explicit_valid = norm_bool(df["has_valid_vote_data"])
    # Only use explicit False as invalid where the original field is not entirely empty/unknown.
    source_not_empty = is_nonempty_text(df["has_valid_vote_data"])
    df.loc[source_not_empty, "has_valid_vote_data_v2"] = explicit_valid[source_not_empty]
    df.loc[source_not_empty, "valid_vote_data_unknown_v2"] = False

note_text = df.get("data_confidence_note", pd.Series("", index=df.index)).fillna("").astype(str).str.lower()
df["explicit_invalid_vote_note_v2"] = note_text.str.contains("no usable valid vote|invalid vote|missing margin|no latest election", regex=True)

df["missing_or_invalid_election_flag_v2"] = (
    ~df["has_election_year_v2"]
    | ~df["has_top_party_v2"]
    | ~df["has_margin_v2"]
    | ~df["has_valid_vote_data_v2"]
    | df["explicit_invalid_vote_note_v2"]
)

# Caveat-type flags.
df["county_election_evidence_flag_v2"] = False
if "county_election_caveat" in df.columns:
    df["county_election_evidence_flag_v2"] |= is_nonempty_text(df["county_election_caveat"])
if "latest_election_aggregation_label" in df.columns:
    df["county_election_evidence_flag_v2"] |= df["latest_election_aggregation_label"].fillna("").astype(str).str.lower().str.contains("ced|county", regex=True)

df["boundary_caveat_flag_v2"] = False
if "boundary_caveat" in df.columns:
    df["boundary_caveat_flag_v2"] |= is_nonempty_text(df["boundary_caveat"])
df["sefton_boundary_flag_v2"] = df.get("LAD25NM", pd.Series("", index=df.index)).fillna("").astype(str).str.lower().eq("sefton")
df["boundary_caveat_flag_v2"] |= df["sefton_boundary_flag_v2"]

df["technical_caveat_flag_v2"] = df["valid_vote_data_unknown_v2"]

# Evidence type.
def evidence_type(row):
    if row["missing_or_invalid_election_flag_v2"]:
        return "missing_or_invalid"
    if row["county_election_evidence_flag_v2"]:
        return "county_derived_or_proxy"
    if row["boundary_caveat_flag_v2"]:
        return "boundary_interim"
    if row["technical_caveat_flag_v2"]:
        return "ward_level_valid_vote_unknown"
    return "ward_level"

df["electoral_evidence_type_v2"] = df.apply(evidence_type, axis=1)

# Confidence band.
def confidence_band(row):
    if row["missing_or_invalid_election_flag_v2"]:
        return "Serious caveat / manual review"
    if row["county_election_evidence_flag_v2"] or row["boundary_caveat_flag_v2"]:
        return "Medium confidence"
    return "High confidence"

df["report_confidence_band_v2"] = df.apply(confidence_band, axis=1)

def caveat_level(row):
    if row["report_confidence_band_v2"] == "Serious caveat / manual review":
        return "serious"
    if row["report_confidence_band_v2"] == "Medium confidence":
        return "medium"
    return "none"

df["report_caveat_level_v2"] = df.apply(caveat_level, axis=1)

# Human-readable summary.
def caveat_summary(row):
    parts = []
    if row["missing_or_invalid_election_flag_v2"]:
        missing = []
        if not row["has_election_year_v2"]: missing.append("missing election year")
        if not row["has_top_party_v2"]: missing.append("missing top party")
        if not row["has_margin_v2"]: missing.append("missing margin")
        if not row["has_valid_vote_data_v2"]: missing.append("invalid valid-vote data")
        if row["explicit_invalid_vote_note_v2"]: missing.append("explicit invalid-data note")
        parts.append("; ".join(missing) if missing else "missing/invalid election evidence")
    if row["county_election_evidence_flag_v2"]:
        parts.append("county-derived/proxy election evidence")
    if row["boundary_caveat_flag_v2"]:
        parts.append("boundary/interim geography caveat")
    if row["technical_caveat_flag_v2"] and not parts:
        parts.append("valid-vote column unavailable in review table; not treated as serious")
    return "; ".join(parts) if parts else "no major caveat"

df["report_caveat_summary_v2"] = df.apply(caveat_summary, axis=1)

# Revised lane: do not turn medium caveats into serious manual-review rows.
def revised_lane(row):
    if row["report_confidence_band_v2"] == "Serious caveat / manual review":
        return "Manual Review / Serious Caveat"
    return row.get("strategic_lane", "Monitor")

df["revised_strategic_lane_v2"] = df.apply(revised_lane, axis=1)
df["include_in_main_report_v2"] = df["report_confidence_band_v2"].ne("Serious caveat / manual review")
df["include_in_serious_caveat_appendix_v2"] = df["report_confidence_band_v2"].eq("Serious caveat / manual review")

display(df["report_confidence_band_v2"].value_counts(dropna=False).reset_index(name="rows"))

,report_confidence_band_v2,rows
0,High confidence,671
1,Medium confidence,153
2,Serious caveat / manual review,1


## 22b.3 Export revised files

In [5]:
summary = (
    df.groupby(["report_confidence_band_v2", "report_caveat_level_v2", "electoral_evidence_type_v2"], dropna=False, as_index=False)
    .agg(
        wards=("WD25CD", "nunique"),
        rows=("WD25CD", "size"),
        mean_score=("initial_watchlist_score", "mean"),
    )
    .sort_values(["report_caveat_level_v2", "electoral_evidence_type_v2"])
)

outputs = {
    "north_west_revised_consolidated_review_v2.csv": df,
    "north_west_caveat_recode_summary_v2.csv": summary,
    "north_west_high_confidence_review_v2.csv": df[df["report_confidence_band_v2"].eq("High confidence")],
    "north_west_medium_confidence_review_v2.csv": df[df["report_confidence_band_v2"].eq("Medium confidence")],
    "north_west_serious_caveat_manual_review_v2.csv": df[df["report_confidence_band_v2"].eq("Serious caveat / manual review")],
    "north_west_reportable_main_review_v2.csv": df[df["include_in_main_report_v2"]],
}

for filename, out_df in outputs.items():
    path = OUTPUT_DIR / filename
    out_df.to_csv(path, index=False)
    print(filename, out_df.shape)

# A one-row audit useful for report notes.
audit = pd.DataFrame([{
    "run_date": datetime.now().isoformat(timespec="seconds"),
    "input_review_file": str(review_path),
    "input_model_file": str(model_input_path) if model_input_path else "not found",
    "total_rows": len(df),
    "high_confidence_rows": int(df["report_confidence_band_v2"].eq("High confidence").sum()),
    "medium_confidence_rows": int(df["report_confidence_band_v2"].eq("Medium confidence").sum()),
    "serious_caveat_rows": int(df["report_confidence_band_v2"].eq("Serious caveat / manual review").sum()),
}])
audit.to_csv(OUTPUT_DIR / "north_west_caveat_recode_audit_v2.csv", index=False)

display(summary)

north_west_revised_consolidated_review_v2.csv (825, 74)
north_west_caveat_recode_summary_v2.csv (4, 6)
north_west_high_confidence_review_v2.csv (671, 74)
north_west_medium_confidence_review_v2.csv (153, 74)
north_west_serious_caveat_manual_review_v2.csv (1, 74)
north_west_reportable_main_review_v2.csv (824, 74)


,report_confidence_band_v2,report_caveat_level_v2,electoral_evidence_type_v2,wards,rows,mean_score
1,Medium confidence,medium,boundary_interim,22,22,48.082857
2,Medium confidence,medium,county_derived_or_proxy,131,131,60.335008
0,High confidence,none,ward_level,671,671,51.945175
3,Serious caveat / manual review,serious,missing_or_invalid,1,1,55.817210
